Two recording folds with fixed peak decoding and one-to-one matching to partial tracking annotations. Silver segmentation supplies pseudo-labels for training.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', 'imagecodecs'], check=True)

import hashlib, json, re, urllib.request, zipfile
from pathlib import Path
import numpy as np
import tifffile
import torch
from torch import nn
from torch.nn import functional as F
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

assert torch.cuda.is_available(), 'Enable Kaggle GPU'
torch.manual_seed(42)
rng = np.random.default_rng(42)
archive = Path('/tmp/ce.zip')
url = 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N3DH-CE.zip'
urllib.request.urlretrieve(url, archive)
with archive.open('rb') as f:
    digest = hashlib.file_digest(f, 'sha256').hexdigest()
assert digest == 'eb3d37cacb3b51d3a427a0c43deaaf1d60cf2eba009f12569e720c2c2fd1e04c', 'Dataset changed'
out = Path('/kaggle/working')
with zipfile.ZipFile(archive) as z:
    def files(pattern):
        return {int(m[1]): name for name in z.namelist() if (m := re.search(pattern, name))}
    def read(name):
        with z.open(name) as f:
            import io
            return tifffile.imread(io.BytesIO(f.read()))
    def normalize(a):
        lo, hi = np.percentile(a, (1, 99.8))
        return np.clip((a.astype(np.float32)-lo)/max(1,hi-lo),0,1)
    def block(a,b):
        return nn.Sequential(nn.Conv3d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU(),
                             nn.Conv3d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU())
    class Detector(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc, self.deep, self.dec, self.head = block(1,8), block(8,16), block(24,8), nn.Conv3d(8,1,1)
        def forward(self,x):
            a=self.enc(x)
            b=self.deep(F.avg_pool3d(a,(1,2,2)))
            return self.head(self.dec(torch.cat([a,F.interpolate(b,size=a.shape[2:],mode='trilinear',align_corners=False)],1)))

    from skimage.feature import peak_local_max
    from scipy.optimize import linear_sum_assignment
    from scipy.spatial.distance import cdist

    def centers(labels):
        ids = np.unique(labels)
        return np.asarray(ndi.center_of_mass(labels > 0, labels, ids[ids > 0])).reshape(-1, 3)

    def train_and_evaluate(train_seq, test_seq):
        raw = files(rf'(?:^|/){train_seq}/t(\d+)\.tif$')
        silver = files(rf'(?:^|/){train_seq}_ST/SEG/man_seg(\d+)\.tif$')
        test = files(rf'(?:^|/){test_seq}/t(\d+)\.tif$')
        truth = files(rf'(?:^|/){test_seq}_GT/TRA/man_track(\d+)\.tif$')
        train_ids, test_ids = sorted(raw.keys() & silver.keys()), sorted(test.keys() & truth.keys())
        assert len(train_ids) == {'01': 195, '02': 190}[train_seq]
        assert len(test_ids) == {'01': 195, '02': 190}[test_seq]
        model = Detector().cuda()
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
        updates = 0
        for epoch in range(1, 21):
            model.train()
            losses = []
            for t in rng.permutation(train_ids):
                image, labels = normalize(read(raw[t])), read(silver[t])
                assert image.shape == labels.shape and image.ndim == 3
                points = centers(labels).round().astype(int)
                if not len(points):
                    continue
                size = np.minimum(image.shape, (16, 64, 64))
                for _ in range(4):
                    start = np.clip(points[rng.integers(len(points))] - size//2 + rng.integers(-6,7,3),
                                    0, np.asarray(image.shape)-size)
                    slices = tuple(slice(int(a), int(a+b)) for a,b in zip(start,size))
                    local = points - start
                    local = local[((local >= 0) & (local < size)).all(axis=1)]
                    target = np.zeros(tuple(size), dtype=np.float32)
                    target[tuple(local.T)] = 1
                    target = ndi.maximum_filter(target, size=(1,3,3))
                    x = torch.as_tensor(image[slices].copy(), dtype=torch.float32, device='cuda')[None,None]
                    y = torch.as_tensor(target, device='cuda')[None,None]
                    logits = model(x)
                    loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=x.new_tensor(30.))
                    assert torch.isfinite(loss)
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    optimizer.step()
                    updates += 1
                    losses.append(loss.detach())
            assert losses
            print(json.dumps(dict(train=train_seq, epoch=epoch, optimizer_steps=updates,
                                  loss=torch.stack(losses).mean().item())), flush=True)
            torch.save(dict(model=model.state_dict(), epoch=epoch, optimizer_steps=updates),
                       out / f'centers_train_{train_seq}.pt')

        model.eval()
        rows = []
        with torch.inference_mode():
            for t in test_ids:
                image, labels = normalize(read(test[t])), read(truth[t])
                assert image.shape == labels.shape
                scores = np.empty_like(image, dtype=np.float32)
                for start in range(0,len(image),8):
                    lo, hi = max(0,start-4), min(len(image),start+12)
                    end = min(len(image),start+8)
                    x = torch.as_tensor(image[lo:hi], dtype=torch.float32, device='cuda')[None,None]
                    scores[start:end] = model(x).sigmoid()[0,0,start-lo:end-lo].cpu().numpy()
                pred = peak_local_max(scores, threshold_abs=0.5, min_distance=2, exclude_border=False)
                actual = centers(labels)
                distances = cdist(actual, pred)
                matches = 0
                if distances.size:
                    penalty = (min(distances.shape)+1)*4
                    i,j = linear_sum_assignment(np.where(distances <= 3, distances, penalty))
                    matches = int((distances[i,j] <= 3).sum())
                rows.append(dict(frame=t, predicted=len(pred), annotated=len(actual), matched=matches))
        return dict(train=train_seq, heldout=test_seq, train_frames=len(train_ids), optimizer_steps=updates,
                    annotated=sum(r['annotated'] for r in rows), matched=sum(r['matched'] for r in rows), frames=rows)

    summary = dict(archive_sha256=digest, supervision='silver segmentation derived centers',
                   match_gate_voxels=3, threshold=0.5, folds=[])
    for train_seq, test_seq in [('01','02'), ('02','01')]:
        summary['folds'].append(train_and_evaluate(train_seq, test_seq))
        (out/'summary.json').write_text(json.dumps(summary,indent=2))
